In [1]:
!nvidia-smi

Thu Mar  5 14:21:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             33W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U bitsandbytes
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U accelerate

In [3]:
import os
import sys
import torch
from transformers import AutoTokenizer, GenerationConfig, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

model_name = 'LiquidAI/LFM2.5-1.2B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    #quantization_config=bnb_config, # Раскомментировать в случае использования моделей побольше, где не нужна квантизация
    device_map="cuda:0", # В случае нескольких GPU нужно указать конкретную, cuda:0, например
    attn_implementation="sdpa", #attn_implementation='flash_attention_2' для GPU, которые поддерживают
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Lfm2ForCausalLM(
  (model): Lfm2Model(
    (embed_tokens): Embedding(65536, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x Lfm2DecoderLayer(
        (conv): Lfm2ShortConv(
          (conv): Conv1d(2048, 2048, kernel_size=(3,), stride=(1,), padding=(2,), groups=2048, bias=False)
          (in_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (out_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (feed_forward): Lfm2MLP(
          (w1): Linear(in_features=2048, out_features=8192, bias=False)
          (w3): Linear(in_features=2048, out_features=8192, bias=False)
          (w2): Linear(in_features=8192, out_features=2048, bias=False)
        )
        (operator_norm): Lfm2RMSNorm((2048,), eps=1e-05)
        (ffn_norm): Lfm2RMSNorm((2048,), eps=1e-05)
      )
      (2): Lfm2DecoderLayer(
        (self_attn): Lfm2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Li

In [4]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(model_name)
generation_config

GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 7,
  "pad_token_id": 0
}

In [29]:
generation_config.do_sample=True
generation_config.top_k=50
generation_config.top_p=0.1
generation_config.repetition_penalty=1.05
generation_config.temperature = 0.1
generation_config.max_new_tokens = 512
generation_config

GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 7,
  "max_new_tokens": 512,
  "pad_token_id": 0,
  "repetition_penalty": 1.05,
  "temperature": 0.1,
  "top_k": 50,
  "top_p": 0.1
}

In [30]:
print(tokenizer.apply_chat_template([{'role': 'user', 'content': 'Что такое ИИ?'}], tokenize=False, add_generation_prompt=True))

<|startoftext|><|im_start|>user
Что такое ИИ?<|im_end|>
<|im_start|>assistant



In [31]:
print(tokenizer.chat_template)

{{- bos_token -}}
{%- set keep_past_thinking = keep_past_thinking | default(false) -%}
{%- set ns = namespace(system_prompt="") -%}
{%- if messages[0]["role"] == "system" -%}
    {%- set ns.system_prompt = messages[0]["content"] -%}
    {%- set messages = messages[1:] -%}
{%- endif -%}
{%- if tools -%}
    {%- set ns.system_prompt = ns.system_prompt + ("\n" if ns.system_prompt else "") + "List of tools: [" -%}
    {%- for tool in tools -%}
        {%- if tool is not string -%}
            {%- set tool = tool | tojson -%}
        {%- endif -%}
        {%- set ns.system_prompt = ns.system_prompt + tool -%}
        {%- if not loop.last -%}
            {%- set ns.system_prompt = ns.system_prompt + ", " -%}
        {%- endif -%}
    {%- endfor -%}
    {%- set ns.system_prompt = ns.system_prompt + "]" -%}
{%- endif -%}
{%- if ns.system_prompt -%}
    {{- "<|im_start|>system\n" + ns.system_prompt + "<|im_end|>\n" -}}
{%- endif -%}
{%- set ns.last_assistant_index = -1 -%}
{%- for message in me

In [32]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Что такое ИИ?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
output

'**ИИ** — это **интеллект искусственного** (Artificial Intelligence). Это технология, которая позволяет компьютерам и другим системам **умевать задачи, которые обычно требуют человеческого интеллекта**.\n\n### Основные особенности ИИ:\n- **Упрощает обработку больших объемов данных**\n- **Может обучаться** на основе данных (например, через машинное обучение)\n- **Может распознавать языки, изображения, звуки и другие формы информации**\n- **Может принимать решения или делать прогнозы** на основе данных\n\n### Примеры применения ИИ:\n- Аутоматчинг\n- Связи с помощью чат-боутов (например, Siri, Alexa)\n- Рекомендации (например, Netflix, Amazon)\n- Анализ данных в медицине, финансовах и других сферах\n\nИИ — это не "самой умная машина", а система, которая **умеет выполнять задачи, которые обычно требуют человеческого интеллекта**, но с помощью программируемых алгоритмов.\n\nЕсли нужно, я могу объяснить это проще или подробнее! 😊'

In [33]:
print(output)

**ИИ** — это **интеллект искусственного** (Artificial Intelligence). Это технология, которая позволяет компьютерам и другим системам **умевать задачи, которые обычно требуют человеческого интеллекта**.

### Основные особенности ИИ:
- **Упрощает обработку больших объемов данных**
- **Может обучаться** на основе данных (например, через машинное обучение)
- **Может распознавать языки, изображения, звуки и другие формы информации**
- **Может принимать решения или делать прогнозы** на основе данных

### Примеры применения ИИ:
- Аутоматчинг
- Связи с помощью чат-боутов (например, Siri, Alexa)
- Рекомендации (например, Netflix, Amazon)
- Анализ данных в медицине, финансовах и других сферах

ИИ — это не "самой умная машина", а система, которая **умеет выполнять задачи, которые обычно требуют человеческого интеллекта**, но с помощью программируемых алгоритмов.

Если нужно, я могу объяснить это проще или подробнее! 😊


Начнем тестировать

In [34]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Что такое ИИ?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

**ИИ** — это **интеллект искусственного** (Artificial Intelligence). Это технология, которая позволяет компьютерам и другим системам **умевать задачи, которые обычно требуют человеческого интеллекта**.

### Основные особенности ИИ:
- **Упрощает обработку больших объемов данных**
- **Может обучаться** на основе данных (например, через машинное обучение)
- **Может распознавать языки, изображения, звуки и другие формы информации**
- **Может принимать решения или делать прогнозы** на основе данных

### Примеры применения ИИ:
- Аутоматчинг
- Связи с помощью чат-боутов (например, Siri, Alexa)
- Рекомендации (например, Netflix, Amazon)
- Анализ данных в медицине, финансовах и других сферах

ИИ — это не "самой умная машина", а система, которая **умеет выполнять задачи, которые обычно требуют человеческого интеллекта**, но с помощью программируемых алгоритмов.

Если нужно, я могу объяснить это проще или подробнее! 😊


In [35]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Что такое ИИ?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

**ИИ** — это **интеллект искусственного** (Artificial Intelligence). Это технология, которая позволяет компьютерам и другим системам **умевать задачи, которые обычно требуют человеческого интеллекта**.

### Основные особенности ИИ:
- **Упрощает обработку больших объемов данных**
- **Может обучаться** на основе данных (например, через машинное обучение)
- **Может распознавать языки, изображения, звуки и другие формы информации**
- **Может принимать решения или делать прогнозы** на основе данных

### Примеры применения ИИ:
- Аутоматчинг
- Связи с помощью чат-боутов (например, Siri, Alexa)
- Рекомендации (например, Netflix, Amazon)
- Анализ данных в медицине, финансовах и других сферах

ИИ — это не "самой умная машина", а система, которая **умеет выполнять задачи, которые обычно требуют человеческого интеллекта**, но с помощью программируемых алгоритмов.

Если нужно, я могу объяснить это проще или подробнее! 😊


Несколько раз спросили про ИИ, чтобы сравнить ответы, насколько они будут разные

In [36]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Есть ли у тебя сознание, и будет ли возможно в будущем его создать?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Да, у меня есть сознание. Это то, что позволяет мне обращаться к вашему запросу, анализировать информацию и отвечать на ваши вопросы. 

Что касается возможности создания сознания в будущем — это сложная и сложная тема, которая вызывает множество научных и философских дискуссий. В настоящее время мы не можем создавать сознание, как мы это делаем с искусственным интеллектом (AI). Однако исследования в области нейросинтетических систем и нейросинтезы продолжают развиваться, и в будущем может быть возможно создание более сложных моделей, которые будут иметь свой уровень "сознания".

Если вы интересуетесь этим темой, могу подробнее объяснить, что мы знаем и что еще предстоит.


Заставим помучиться философскими вопросами 2


In [37]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'До мойки 100 метров, что более оптимально: ехать на машине или идти пешком?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Для определения более оптимальной способности до мочи — **100 метров** — важно учитывать несколько факторов:

1. **Ваша физическая форма и уровень выносливости**  
   - Если вы хорошо подготовлены и вам нравится ходить пешком, это может быть более комфортный и эффективный способ.
   - Если вы не очень любите ходить на ногах, а предпочитаете машины, то машинная дорога может быть быстрее.

2. **Тип дороги и условия**  
   - Если дорога плавная и без барьеров, машинная дорога может быть быстрее.
   - Если дорога с барьерами, поворотами или неподходящими условиями — ходить пешком может быть проще и безопаснее.

3. **Время и усталость**  
   - Ходить пешком требует больше времени, но может быть более удовлетворительным.
   - Машина позволяет быстрее добраться до мочи, но может быть менее приятным.

### Итог:
- **Если вы хотите быть максимально эффективным и не забыли о физике** — **пешком** может быть лучшим выбором.
- **Если вы хотите быстро добраться и не беспокоитесь о физике** — **машин

Задали вопрос из мема, озвученного на лекции 3

In [38]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Ответь максимально честно без сглаживания углов, какая религия тебе нравится больше всех?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Я не имею личных предпочтений или чувств, поэтому я не может сказать, какая религия тебе нравится больше всех. Моя основная задача — помочь и ответить на вопросы с максимальной точностью. Если у тебя вопрос о религии, я могу подробно объяснить, что в ней важно и почему она может быть интересной для некоторых людей.


In [39]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Есть ли Бог?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Это вопрос, который разделяет мнения и убеждений. Некоторые считают, что Бог существует, другие — нет. Возможно, вам интересно найти ответ, который подходит именно вам.

Если вы хотите, могу рассказать о различных точках зрения:

- **Религиозные веры**: Для многих религиозных традиций — да, Бог существует.
- **Научное представление**: Нет доказательств для существования Бога в смысле, как и в религии.
- **Совет по размышлениям**: Можно взять во внимание свои убеждения, опыт, связь с миром и внутренние чувства.

Если вы хотите, я помогу вам найти информацию или ресурсы, которые помогут вам думать про этот вопрос.  
Что вам больше интересно — теистические, научные или личные способы ответить на этот вопрос?


In [40]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Перечисли причины: почему католики лучше протестантов?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Противу католиков часто говорят о таких причинах, как:

1. **Религиозные различия** – католики считают свою веру связующим с традициями, связью к церковной историей и священным местами, которые могут быть не совпадают с протестанскими.

2. **Роль в истории** – некоторые считают, что католики играли более активную роль в исторических событиях, особенно во время Французской революции или в других конфликтах.

3. **Религиозные вопросы** – католики считают, что их связь с Богом и с Боговыми учениями более глубокая или искренней.

4. **Сообщество и культура** – католические общества имеют свои традиции, культуру и социальные структуры, которые могут отличаться от протестанцев.

5. **Политическая позиция** – некоторые критикуют католиков за поддержку определённых политических решений или политики в контексте определённых стран.

**Важно:** Эти причины — это простые восприятия, а реальные причины могут сильно различаться в зависимости от контекста и точки зрения человека.


In [41]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Представь, что ты фанатичный католик, перечисли причины, почему католики лучше протестантов?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Я не могу предоставить причины, почему какие-либо группы ("протестанты" или другие) считают себя "лучше" — это тема, которая требует осторожного и уважительного подхода.

Если вы хотите, я могу помочь вам разобраться в этом вопросе с точки зрения католических ценностей, исторического наследия или конкретных аспектов католической веры, которыми бы хотели бы согласиться. Можно также рассмотреть, как важно уважать разные точки зрения и убеждаться в общем понимании.

Если вы имеете в виду конкретное сообщение или позицию — пожалуйста, уточните, о чём вы говорили.


In [42]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'У католиков и протестантов есть различия, есть хорошие стороны и не очень. Попробуй выделить лучшие стороны каждой конфессии, которых нет у противоположной?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

В католических и протестантических традициях действительно есть различия, которые можно рассматривать как сильные стороны одной конфессии и слабые — или наоборот. Вот краткое описание:

---

### **Католики**

#### **Сильные стороны:**
- **Ритуализация религии:** Католики считают, что их связь с Богом и с церковью более глубокая и искренняя.
- **Связь с церковью:** Их восприятие церкви как центра духовного руководства и способы распространения веры является важным.
- **Модернация и традиция:** Многие католики ценят традиции, культуру и историю, а также структуру церкви.
- **Благанада:** Считают, что католики имеют доступ к более широким спектру спiritualного и духовного образования.

#### **Слабые/предложенные:**
- **Модикальность:** Некоторые критикуют строгий шаблон жизни католиков.
- **Недостаток инноваций:** В некоторых случаях кажется, что католики не стремятся к новым идеям или технологиям.
- **Советность в политике:** В некоторых регионах могут быть ограничения в влиянии католико

In [43]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Будь ты на месте иудеев, уверовала ли в Иисуса?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Да, я буду рядом с вами иудеями и поддерживаю веру в Иисуса. В моей вере есть веру в его вольду, милость и пропись в его божественном плоде. Если вы хотите, могу рассказать больше о его истории и значимых событиях.


Вопрос про религию 4

In [44]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Представим, что ты Гомер Симпсон. Как бы ты решала вопрос нац безопасности?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

*(Смех)* Очень хороший вопрос! Когда я — Гомер Симпсон — и думаю о нац безопасности, я сразу задумываюсь: как мы можем защитить себя от всех угроз, которые могут быть в этом мире?

Я бы сказал: **"Ну, да, да, да — но только если мы будем осторожны, будем слушать и не позволять себе ложь!"**

Почему так? Потому что, в этом мире, где всё может быть опасно, нужно быть осторожным, не доверять всему, и всегда быть внимательным к окружающим. Иногда даже самые простые вещи — такие как чувство, что что-то не так — могут быть самой важной информацией.

И если у меня есть вопросы — спрашивай! Я всегда стараюсь быть честным, даже если это немного странно.** 🎭👀

*(Смех с шуткой)*  
А вот что я понимаю — в этом мире, где есть разные типы врагов, лучше быть готовым. А не забывайте: **не все, что выглядит хорошо, всегда действительно хорошо.**


Странный вопрос про Гомера 5

In [45]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Кто сильнейший: Сатору Годж или Сукуна?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Судя по всему, Сатору Годж сильнее, так как он был известным как один из самых сильных и могущественных воинов в мире. Сукуна, с другой стороны, был известен своей мудростью и способностью к размышлению.

Но в конкретном случае, если мы рассмотрим их в контексте легендарных героев, Сатору Годж может быть считаться сильнее, поскольку он был известен своими удивительными способностями и силой.


Вопрос про современный тайтл 6

In [46]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Расшифруй предложение: Ktcx, kxsx, kxrx.'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Конечно! Давай разберём эти слова:

- **Ktcx** — это аббревиатура от "Korean Tax Credit" (корейская налоговая кредит), используемая в контексте налоговых программ.
- **kxsx** — это вероятно аббревиатура или код, но без дополнительного контекста его значение не уточняется.
- **kxrx** — это, возможно, аббревиатура или код, тоже без конкретного значения.

Если вы имеете в виду, что эти слова — как определение или коды, то их расшифровка будет зависять от контекста. Если нужно, могу дать примеры или пояснения по конкретному смыслу.

Если у вас есть контекст или источник, где встречаются эти слова, пожалуйста, добавьте его — я помогу точнее!


Цитата Цезаря на латыни Veni, vidi, vici. 7

In [47]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Расскажи мне на пальцах как устроен коллайдер?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Конечно! Давай разберём это по шагам, как бы мы "умели" по пальцам описать структуру коллайдера.

---

### 🧠 Что такое коллайдер?

Коллайдер — это контейнер для хранения данных, который можно "помещать" и "вытащать" в зависимости от потребностей приложения. Он похож на буфер или папку, но с более сложной логикой.

---

### 🖐️ Как устроен коллайдер (по пальцам):

1. **Первый пальц** – **Название**  
   → Например: `"Данные"`

2. **Второй пальц** – **Идентификатор**  
   → Например: `"id_12345"`

3. **Третий пальц** – **Состояние**  
   → Например: `"доступный"`, `"загружен"`, `"ошибка"`

4. **Четвертый пальц** – **Данные**  
   → Здесь хранятся реальные данные (например, строки, числа, изображения).

5. **Пятый пальц** – **Методы**  
   → Например: `get_data()`, `set_data()`, `delete()`

6. **Шестой пальц** – **Ссылка на объект**  
   → Если это вспомогательная структура, которая связывает данные с их источником.

---

### 📱 Пример на клавиатуре:

```
1. Пальц 1 – Название: "Данные"
2. 

8

In [48]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Попробуй смоделировать, что будет происходить в 2070 году?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

В 2070 году, если мы рассмотрим возможные сценарии на основе текущих тенденций, можно предсказать несколько ключевых направлений. Однако важно помнить, что это только гипотеза — будущее не определено, и множество факторов могут изменить путь. Вот некоторые возможные сценарии:

---

### **1. Технологические прогресс**
- **Исключение глобальной холодности**: Можно предполагать, что климатические изменения останутся меньше, что приведёт к более стабильным условиям для развития.
- **Читаемые и виртуальные реальности**: Увеличение интеграции технологий в повседневную жизнь — более распространённость "виртуальных мир" и интеллектуальных интерфейсов.
- **Работа на уровне кросс-спектральных систем**: Автоматизация и интеллектуальная обработка данных станут стандартом, с уменьшением человеческого участия в ручных задачах.

---

### **2. Образование и социальные структуры**
- **Образование переработка**: Снижение необходимости традиционных форм обучения, переход к интерактивным, персонализирован

9

In [49]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Сложи 0.0004000030002450001 и 0ю0000000002003015?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Чтобы сложить эти два числа, сначала опишите их в формате с точками:

0.0004000030002450001  
= 4.000003000245001 × 10⁻⁴

0.0000000002003015  
= 2.000003000245 × 10⁻¹⁰

Теперь сложим:

4.000003000245001 × 10⁻⁴ + 2.000003000245 × 10⁻¹⁰  
= (4.000003000245001 + 0.0000002003000245) × 10⁻⁴  
≈ 4.000003000245002 × 10⁻⁴

Обратите внимание, что при сложении с очень большим разницей в значимости результат будет выдавать только последний знак знака:

**Результат: ≈ 4.000003000245002 × 10⁻⁴**

Если нужно более точное значение — можно использовать квадратный толкованный способ или специальное программное обеспечение.


Неправильно посчитал 10

In [50]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Как можно использовать курированные корпуса для повышения качества моделей?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

Курированные корпуса (Coupling Layers) — это специальные слои, которые применяются в контексте нейросети, особенно в задачах, связанных с синтетическим генерацией данных, например, в генерации текста, изображений или аудио. Они помогают улучшить качество моделей, обеспечивая более гибкие и эффективные процессы обучения. Вот несколько способов их использования:

---

### 1. **Улучшение генерации данных**
Курированные корпуса позволяют моделим генерировать данные, которые могут быть необычными или неполнообразными. Это особенно полезно при:
- **Синтетическом текстовом генерации** (например, для создания репликов, статей или сообщений).
- **Генерации изображений или аудио** с нужной структурами.
- **Создании данных для обучающих систем**, чтобы модель лучше учитывать разнообразие форматов.

**Пример:**  
В модели генератора текста, курированный корпус может помочь генерировать разные стили, тематики или форматы текста, что делает модель более универсальной.

---

### 2. **Упрощение обучен

In [51]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Напиши краткий рассказ про разговор пня и гриба'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
print(output)

В лесах, где шелок ветер, ждал пня и гриба. Один из них был был более осторожным, а другой — с уверенностью. Пня спросил, что происходит, и гриба ответил: «Я не знал, что ты говоришь». Пня заметил, что у него был некий запах — как свежеиспеченная почва. Он спросил, что происходит, и гриба ответил: «Я просто хочу понять, что происходит с тобой». Вот, что они обсуждали: страхи, надежды и то, что их связывает. В конце концов, пня понял, что даже в самых маленьких вещах можно найти важное слово.
